In [3]:
import openmeteo_requests
import requests_cache
import pandas as pd
import json
import os
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# Coordinates for postal codes
postal_codes = {
    "2000": (51.219464, 4.397171),
    "2060": (51.233034, 4.441093),
    "2018": (51.215499, 4.404720),
    "2020": (51.209478, 4.398984),
    "2030": (51.217778, 4.448460),
    "2050": (51.229574, 4.402569),
    "2100": (51.209201, 4.370597),
    "2140": (51.219432, 4.497435),
    "2170": (51.224212, 4.494332),
    "2600": (51.220976, 4.400131),
    "2610": (51.229000, 4.340580),
    "2660": (51.215189, 4.220453)
}

# Create directory for weather data if it doesn't exist
output_dir = "weatherdata"
os.makedirs(output_dir, exist_ok=True)

# Define the last month before 2019/09/22 (August 2019)
start_date = "2023-08-15"
end_date = "2023-09-15"

# Fetch weather data for each postal code and save it
for postal_code, (latitude, longitude) in postal_codes.items():
    # API Request
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ["temperature_2m", "precipitation"]
    }

    # Get weather data
    responses = openmeteo.weather_api(url, params=params)

    # Initialize the list to collect all hourly data
    all_weather_data = []

    # Process each location's response
    for response in responses:
        hourly = response.Hourly()
        hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
        hourly_precipitation = hourly.Variables(1).ValuesAsNumpy()

        # Create timestamps for each hourly entry
        hourly_timestamps = pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        )

        # Collect data for each hour
        for i, timestamp in enumerate(hourly_timestamps):
            data = {
                "timestamp": timestamp.isoformat(),
                "latitude": latitude,
                "longitude": longitude,
                "temperature_2m": float(hourly_temperature_2m[i]),
                "precipitation": float(hourly_precipitation[i])
            }
            all_weather_data.append(data)

    # Save the collected data for the postal code as a single JSON file
    file_name = f"{postal_code}.json"
    file_path = os.path.join(output_dir, file_name)  # Corrected line

    with open(file_path, "w") as json_file:
        json.dump(all_weather_data, json_file, indent=4)

    print(f"Weather data for postal code {postal_code} saved in {file_path}.")

print("All weather data has been saved.")


Weather data for postal code 2000 saved in weatherdata\2000.json.
Weather data for postal code 2060 saved in weatherdata\2060.json.
Weather data for postal code 2018 saved in weatherdata\2018.json.
Weather data for postal code 2020 saved in weatherdata\2020.json.
Weather data for postal code 2030 saved in weatherdata\2030.json.
Weather data for postal code 2050 saved in weatherdata\2050.json.
Weather data for postal code 2100 saved in weatherdata\2100.json.
Weather data for postal code 2140 saved in weatherdata\2140.json.
Weather data for postal code 2170 saved in weatherdata\2170.json.
Weather data for postal code 2600 saved in weatherdata\2600.json.
Weather data for postal code 2610 saved in weatherdata\2610.json.
Weather data for postal code 2660 saved in weatherdata\2660.json.
All weather data has been saved.
